In [23]:
# import dependencies
import numpy as py
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
# import data
enrol_path = "C:/Users/ywa/OneDrive - Stifterverband/Dokumente/Data/Enrol/Nov2026/Individual_enrol.csv"

enrol_df = pd.read_csv(enrol_path)

master_path = "C:/Users/ywa/OneDrive - Stifterverband/Dokumente/Data/UserReport/MasterList_Update.csv"

master_df = pd.read_csv(master_path)

hei_path = "C:/Users/ywa/OneDrive - Stifterverband/Dokumente/Data/UserReport/german-heis.xlsx"

hei_df = pd.read_excel(hei_path, sheet_name = 'Final_sheet', usecols = ["Name", "clean_domain"])

# inspeact the data
#enrol_df.head()
master_df.info()
#hei_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28171 entries, 0 to 28170
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   domain         28171 non-null  object 
 1   country        26878 non-null  object 
 2   provider_type  28171 non-null  float64
 3   provider       28170 non-null  object 
 4   count          2310 non-null   float64
dtypes: float64(2), object(3)
memory usage: 1.1+ MB


In [25]:
# courses and institutions
course_list = ["KI-Kompetenzen in der Hochschulverwaltung", 
               "EU AI Act Essentials", 
               "KIÖV - KI in öffentlichen Verwaltungen", 
               "Crafting Precision Prompts with GenAI",
               "KI und Arbeitswelt",
               "Data Literacy - Daten interpretieren durch Data Mining", 
               "KI und Leadership",
               "KI und Leadership - Mikrokurs",
              ]


In [ ]:
# filter courses based on the course_list
selected_courses = enrol_df.loc[enrol_df["Course full name with link"].isin(course_list)]

print(selected_courses["Course full name with link"].unique())

selected_courses.head()


In [ ]:
# strip email domains
# domain regex for extraction
domain_regex = r'(?<=@)[^@\s]+'

# strip the strings xyz@
sachsen_regex = selected_courses["Email address"].astype(str).str.findall(domain_regex)
selected_courses["Email domains"] = [item[0] if item else None for item in sachsen_regex]
selected_courses["Email domains"] = selected_courses["Email domains"].str.lower()
selected_courses["Email domains"] = selected_courses["Email domains"].str.strip().str.replace(r'\s+', '', regex=True).str.normalize('NFKC')
selected_courses.head()

In [35]:
# match given domains and institution names

domains_list = ["uni-leipzig.de",
                "tu-freiberg.de",
                "tu-chemnitz.de",
                "dhsn.de",
                "fh-zwickau.de",
                "whz.de", 
                "hszg.de", 
                "hs-mittweida.de", 
                "htw-dresden.de",
                "htwk-leipzig.de",
                "palucca.eu",
                "hmt-leipzig.de",
                "hfmdd.de",
                "hfbk-dresden.de",
                "hgb-leipzig.de",
               ]

hei_list = ["Universität Leipzig",
            "TU Bergakademie Freiberg",
            "TU Chemnitz", 
            "Duale Hochschule Sachsen"
            "Festsächsische Hochschule Zwickau (FH)", 
            "Westsächsische Hochschule Zwickau",
            "Hochschule Zittau/Görlitz",
            "Hochschule Mittweida",
            "HTW Dresden",
            "HTWK Leipzig",
            "Palucca Hochschule für Tanz",
            "HMT Leipzig",
            "Hochschule für Musik Dresden",
            "HFBK Dresden",
            "HGB Leipzig",
           ]

In [ ]:
# match domains to institution names
matched_enrol = selected_courses.merge(master_df, left_on = "Email domains", right_on = "domain")

matched_enrol.head()


In [41]:
# select matched enrol from certain universities
sachsen_matched = matched_enrol.loc[matched_enrol["provider"].isin(hei_list)]

sachsen_matched = sachsen_matched[["Course full name with link", "Email domains", "provider"]]

sachsen_matched.head()

,Course full name with link,Email domains,provider
371,Crafting Precision Prompts with GenAI,htw-dresden.de,HTW Dresden
510,Crafting Precision Prompts with GenAI,uni-leipzig.de,Universität Leipzig
1355,Crafting Precision Prompts with GenAI,htw-dresden.de,HTW Dresden
1475,Crafting Precision Prompts with GenAI,informatik.tu-chemnitz.de,TU Chemnitz
1509,Crafting Precision Prompts with GenAI,medizin.uni-leipzig.de,Universität Leipzig


In [46]:
# count numbers of users per institute
sachsen_users = sachsen_matched["provider"].value_counts()

sachsen_users

provider
Universität Leipzig                  35
Hochschule Mittweida                 16
HTW Dresden                          13
Hochschule Zittau/Görlitz            10
HTWK Leipzig                          4
TU Chemnitz                           3
Westsächsische Hochschule Zwickau     2
TU Bergakademie Freiberg              2
HMT Leipzig                           1
Name: count, dtype: int64